# Predicting NBA All-Star Selections from Regular-Season Per-Game Stats
**DTSC 2301 – Portfolio Project Two**

**Goal:** Build and evaluate machine-learning models that predict whether a player-season ends in an NBA All-Star selection, using only per-game box-score statistics and position.

**How to run:** download the Kaggle dataset *NBA Stats (1947-present)* by Sumitro Datta (data from Basketball-Reference), and place `Player Per Game.csv` and `All-Star Selections.csv` in a folder named `data/` next to this notebook. Then run all cells top to bottom.

Sections follow the assignment: 1 Problem · 2 Background · 3 Data · 4 EDA · 5 Preparation · 6 Baseline & Models · 7 Evaluation · 8 Interpretation · 9 Limitations · 10 Transparency.

## 1. Problem Definition
- **Prediction problem:** Will a given NBA player be selected as an All-Star in a given season?
- **Target variable:** `all_star` (1 = selected, 0 = not selected)
- **Task type:** Binary **classification** (strongly imbalanced: only about 5% of player-seasons are All-Stars)
- **Unit of analysis:** one player in one regular season
- **Who benefits:** analysts, fans, and front offices who want a stats-based "who deserves it" benchmark, and anyone studying how closely fan/coach voting tracks measurable production.

## 2. Background and Context
See the written post for the sources and APA citations. Key idea used here: era-adjusted, per-minute production matters more than raw totals, so I engineer within-season features.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_curve)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_DIR = Path("data")
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## 3. Data Description
Source: Basketball-Reference via Kaggle (Datta, *NBA Stats (1947-present)*). Two files are used:
- `Player Per Game.csv`: one row per player per team per season (per-game averages).
- `All-Star Selections.csv`: one row per All-Star selection (used only to build the target label).

In [ ]:
def clean_cols(df):
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace("%", "_percent", regex=False)
                  .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
    df.columns = [("x" + c) if c[0].isdigit() else c for c in df.columns]
    df.columns = [c.replace("_per_game", "") for c in df.columns]   # pts_per_game -> pts
    return df

pg  = clean_cols(pd.read_csv(DATA_DIR / "Player Per Game.csv"))
asg = clean_cols(pd.read_csv(DATA_DIR / "All-Star Selections.csv"))
print("Per-game raw shape:", pg.shape)
print("All-Star raw shape:", asg.shape)
print(list(pg.columns))
print(list(asg.columns))
pg.head(3)

## 4. Data Understanding and Exploration (first look)
Check size, missing values, and the structure of traded players (who appear on several rows in one season) before deciding how to clean.

In [ ]:
print(pg.isna().sum()[lambda s: s > 0].sort_values(ascending=False).head(15))
print("\nSeasons covered:", pg["season"].min(), "to", pg["season"].max())
print("Leagues:", pg["lg"].value_counts().to_dict())
print("Example team codes for traded players:", sorted(pg["tm"].astype(str).unique())[:5], "...")

## 5. Data Preparation and Feature Selection
Decisions (justified in the written post):
1. **NBA only** (drop ABA/BAA), because All-Star rules and style of play differ.
2. **Start at the 1981-82 season**, since games started (`gs`) is not recorded earlier and the 3-point line arrived in 1979-80.
3. **Traded players:** keep only the season-total row (`TOT`/`2TM`/`3TM`) so each player-season appears once and stats are not split across teams.
4. **Drop seasons with no All-Star Game** (e.g. 1999 lockout); otherwise every player would be falsely labelled 0.
5. **Minimum 20 games played** to remove tiny-sample noise.
6. **Label:** joined from the All-Star file (replacement selections count as selected).

In [ ]:
df = pg[pg["lg"] == "NBA"].copy()
df = df[df["season"] >= 1982]

tm = df["tm"].astype(str).str.upper()
is_total_row = tm.eq("TOT") | tm.str.fullmatch(r"\d+TM")
key = ["player_id", "season"] if ("player_id" in df.columns and "player_id" in asg.columns) else ["player", "season"]
df = df[is_total_row | ~df.duplicated(key, keep=False)].copy()
assert not df.duplicated(key).any(), "Duplicate player-seasons remain"

asg_nba = asg[asg["lg"].astype(str).str.upper().eq("NBA")] if "lg" in asg.columns else asg
labels = asg_nba[key].drop_duplicates().assign(all_star=1)
df = df.merge(labels, on=key, how="left")
df["all_star"] = df["all_star"].fillna(0).astype(int)

# drop seasons with no All-Star Game
n_per_season = df.groupby("season")["all_star"].sum()
bad = n_per_season[n_per_season == 0].index.tolist()
print("Seasons dropped (no All-Stars recorded):", bad)
df = df[~df["season"].isin(bad)]

df = df[df["g"] >= 20].copy()
df["pos_main"] = df["pos"].astype(str).str.split("-").str[0]
print("Rows after cleaning:", len(df), "| All-Star rate:", round(df["all_star"].mean(), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df["all_star"].value_counts().sort_index()
axes[0].bar(["Not All-Star", "All-Star"], counts.values, color=["#9aa5b1", "#e8590c"])
axes[0].set_title("Class balance (player-seasons)"); axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values): axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")
df.groupby("season")["all_star"].sum().plot(ax=axes[1], color="#e8590c")
axes[1].set_title("All-Stars per season in data"); axes[1].set_ylabel("Players"); axes[1].set_xlabel("Season (end year)")
plt.tight_layout(); save(fig, "01_class_balance"); plt.show()

In [ ]:
stat_cols = [c for c in ["age","g","gs","mp","fg","fga","fg_percent","x3p","x3pa","x3p_percent",
                         "x2p","x2pa","x2p_percent","e_fg_percent","efg_percent","ft","fta","ft_percent",
                         "orb","drb","trb","ast","stl","blk","tov","pf","pts"] if c in df.columns]
print(df[stat_cols].describe().T.round(2))

compare = ["pts", "mp", "ast", "trb", "stl", "blk"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axes.ravel(), compare):
    sns.boxplot(data=df, x="all_star", y=c, ax=ax, palette=["#9aa5b1", "#e8590c"], showfliers=False)
    ax.set_xticklabels(["Not All-Star", "All-Star"]); ax.set_xlabel(""); ax.set_title(f"{c.upper()} per game")
plt.suptitle("All-Stars separate from the field on usage and production", y=1.02)
plt.tight_layout(); save(fig, "02_stat_distributions"); plt.show()

In [ ]:
corr = df[stat_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax, cbar_kws={"label": "Pearson r"})
ax.set_title("Correlation between per-game features")
plt.tight_layout(); save(fig, "03_correlation"); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
         .loc[lambda s: s.abs() > 0.9].sort_values(ascending=False))
print("Highly correlated pairs (|r| > 0.9):\n", pairs.round(3))

print("\nAll-Star rate by position:\n", df.groupby("pos_main")["all_star"].mean().round(3))

**Feature selection.** Many raw columns are near-duplicates (`fg` = `x2p` + `x3p`; `trb` = `orb` + `drb`; `ft` is almost `fta` times `ft_percent`). I keep one version of each to reduce redundancy and keep coefficients interpretable. I also engineer era-aware features:
- `g_frac`: games played divided by the max games in that season (handles shortened seasons).
- `pts_per36`: scoring rate per 36 minutes.
- `pts_rank`, `ast_rank`, `trb_rank`: percentile rank *within the same season* (a 20 PPG season means different things in 1985 vs 2015).

These use only features, never labels, so they do not leak the target.

In [ ]:
drop_redundant = ["fg","x2p","x2pa","x2p_percent","ft","trb","x3p","e_fg_percent","efg_percent"]
df["g_frac"]    = df["g"] / df.groupby("season")["g"].transform("max")
df["pts_per36"] = df["pts"] / df["mp"].clip(lower=1) * 36
for c in ["pts", "ast", "trb"]:
    if c in df.columns:
        df[f"{c}_rank"] = df.groupby("season")[c].rank(pct=True)

numeric_features = [c for c in stat_cols if c not in drop_redundant] + ["g_frac","pts_per36","pts_rank","ast_rank","trb_rank"]
numeric_features = list(dict.fromkeys(numeric_features))
categorical_features = ["pos_main"]
features = numeric_features + categorical_features
print(len(features), "features:", features)

### Train/test strategy and leakage prevention
- **Temporal holdout:** train on older seasons, test on the **most recent 8 seasons**. This mimics real use (predicting future seasons) and avoids the same player's adjacent seasons appearing on both sides of a random split.
- **Cross-validation inside training data** uses `GroupKFold` grouped by season, so a season is never split between fit and validation folds.
- **All preprocessing (imputing, scaling, encoding) lives inside the sklearn `Pipeline`**, so it is fit only on training folds.
- **Excluded on purpose:** player name, ID, team, awards, and All-Star votes, which would leak or memorize the answer.

In [ ]:
test_seasons = [int(s) for s in sorted(df["season"].unique())[-8:]]
train = df[~df["season"].isin(test_seasons)].copy()
test  = df[df["season"].isin(test_seasons)].copy()
X_train, y_train, g_train = train[features], train["all_star"], train["season"]
X_test,  y_test           = test[features],  test["all_star"]
print(f"Train: {len(train):,} rows, seasons {train.season.min()}-{train.season.max()}, All-Star rate {y_train.mean():.3f}")
print(f"Test : {len(test):,} rows, seasons {test.season.min()}-{test.season.max()}, All-Star rate {y_test.mean():.3f}")

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

## 6. Baseline and Model Development
**Baselines**
1. *Majority class* (`DummyClassifier`): always predicts "not All-Star". Shows why accuracy alone is misleading.
2. *Domain rule*: in each season, label the top-K scorers (K = average All-Stars per training season) as All-Stars. This is the "just pick the best scorers" heuristic any model should beat.

**Models:** Logistic Regression (interpretable, linear), Random Forest (non-linear, robust), and Histogram Gradient Boosting (strong tabular learner). Each is tuned with the same grouped CV and the same scoring metric (average precision) for a fair comparison.

In [ ]:
# --- Baselines ---
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
K = int(round(train.groupby("season")["all_star"].sum().mean()))
def top_k_rule(frame, k=K, col="pts"):
    return frame.groupby("season")[col].rank(ascending=False, method="first").le(k).astype(int)
rule_pred = top_k_rule(test)
print("K used for rule baseline:", K)

In [ ]:
cv = GroupKFold(n_splits=5)
search_spaces = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"clf__C": [0.01, 0.1, 1, 10], "clf__class_weight": [None, "balanced"]}),
    "Random Forest": (RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        {"clf__max_depth": [6, 12, None], "clf__min_samples_leaf": [1, 5],
         "clf__class_weight": [None, "balanced_subsample"]}),
    "Gradient Boosting": (HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {"clf__learning_rate": [0.03, 0.1], "clf__max_leaf_nodes": [8, 31],
         "clf__class_weight": [None, "balanced"]}),
}
fitted, cv_scores = {}, {}
for name, (est, grid) in search_spaces.items():
    pipe = Pipeline([("prep", preprocess), ("clf", est)])
    gs = GridSearchCV(pipe, grid, scoring="average_precision", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train, groups=g_train)
    fitted[name], cv_scores[name] = gs.best_estimator_, gs.best_score_
    print(f"{name:20s} CV PR-AUC = {gs.best_score_:.3f}  best params: {gs.best_params_}")

## 7. Model Evaluation and Selection
**Metrics:** because only ~5% of rows are positives, I use **precision** (of players predicted All-Star, how many were?), **recall** (of real All-Stars, how many did we find?), **F1** (balance of the two), **PR-AUC** (ranking quality on the rare class; more informative than ROC-AUC under imbalance), and **ROC-AUC** for reference. Accuracy is shown only to demonstrate that it is misleading here.

**Threshold:** each model's decision threshold is chosen to maximize F1 on *out-of-fold training predictions*, never on the test set.

In [ ]:
def metrics_row(y_true, pred, proba=None):
    row = {"Accuracy": accuracy_score(y_true, pred),
           "Precision": precision_score(y_true, pred, zero_division=0),
           "Recall": recall_score(y_true, pred, zero_division=0),
           "F1": f1_score(y_true, pred, zero_division=0)}
    row["PR-AUC"] = average_precision_score(y_true, proba) if proba is not None else np.nan
    row["ROC-AUC"] = roc_auc_score(y_true, proba) if proba is not None else np.nan
    return row

def best_threshold(model, X, y, groups):
    oof = cross_val_predict(model, X, y, groups=groups, cv=GroupKFold(5), method="predict_proba")[:, 1]
    p, r, t = precision_recall_curve(y, oof)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-9, None)
    return float(t[np.argmax(f1)])

rows, probas, thresholds = {}, {}, {}
rows["Baseline: majority class"] = metrics_row(y_test, dummy.predict(X_test))
rows[f"Baseline: top-{K} scorers/season"] = metrics_row(y_test, rule_pred)
for name, model in fitted.items():
    thr = best_threshold(model, X_train, y_train, g_train)
    proba = model.predict_proba(X_test)[:, 1]
    probas[name], thresholds[name] = proba, thr
    rows[name] = metrics_row(y_test, (proba >= thr).astype(int), proba)

results = pd.DataFrame(rows).T.round(3)
results["CV PR-AUC (train)"] = pd.Series(cv_scores).round(3)
print("Held-out test seasons:", test_seasons)
results

In [ ]:
# Per-season ranking check: pick the top-K players by predicted probability each season
def precision_at_k(frame, proba, k=K):
    tmp = frame[["season", "all_star"]].assign(p=proba)
    hits = tmp.groupby("season").apply(lambda d: d.nlargest(k, "p")["all_star"].mean())
    return hits.mean()
pk = {n: precision_at_k(test, p) for n, p in probas.items()}
pk[f"Baseline: top-{K} scorers/season"] = precision_at_k(test, test["pts"].values)
print(pd.Series(pk, name=f"Precision@{K} per season").round(3))

best_name = max(cv_scores, key=cv_scores.get)   # selected on training CV, not on test
print("\nFinal model (best CV PR-AUC):", best_name)

fig, ax = plt.subplots(figsize=(7, 5))
for name, p in probas.items():
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax.plot(rc, pr, label=f"{name} (AP={average_precision_score(y_test, p):.2f})")
ax.axhline(y_test.mean(), ls="--", color="gray", label=f"Chance ({y_test.mean():.2f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Precision-recall curves on held-out seasons")
ax.legend(); plt.tight_layout(); save(fig, "04_pr_curves"); plt.show()

## 8. Model Interpretation and Insights
Confusion matrix and error analysis for the final model, then which features drive predictions (permutation importance on the test seasons; logistic-regression coefficients for direction of effect).

In [ ]:
final = fitted[best_name]; thr = thresholds[best_name]
test = test.assign(proba=probas[best_name], pred=(probas[best_name] >= thr).astype(int))

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, test["pred"]), display_labels=["Not All-Star", "All-Star"]).plot(ax=ax, cmap="Oranges", colorbar=False)
ax.set_title(f"{best_name}: confusion matrix (test seasons)"); plt.tight_layout(); save(fig, "05_confusion"); plt.show()

imp = permutation_importance(final, X_test, y_test, scoring="average_precision",
                             n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp_df = pd.Series(imp.importances_mean, index=features).sort_values().tail(12)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp_df.index, imp_df.values, color="#e8590c")
ax.set_title(f"Top features: permutation importance ({best_name})"); ax.set_xlabel("Drop in PR-AUC when shuffled")
plt.tight_layout(); save(fig, "06_importance"); plt.show()

In [ ]:
# Direction of effects from the (tuned) logistic regression
lr = fitted["Logistic Regression"]
names = lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=names).sort_values()
show = pd.concat([coefs.head(6), coefs.tail(8)])
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(show.index, show.values, color=["#4c6ef5" if v < 0 else "#e8590c" for v in show.values])
ax.set_title("Logistic regression coefficients (standardized features)"); ax.axvline(0, color="black", lw=0.8)
plt.tight_layout(); save(fig, "07_coefficients"); plt.show()

cols = [c for c in ["player", "season", "pos_main", "age", "g", "mp", "pts", "trb", "ast", "proba"] if c in test.columns]
print("FALSE POSITIVES (predicted All-Star, were not):")
print(test[(test.pred == 1) & (test.all_star == 0)].sort_values("proba", ascending=False)[cols].head(10).round(2).to_string(index=False))
print("\nFALSE NEGATIVES (real All-Stars the model missed):")
print(test[(test.pred == 0) & (test.all_star == 1)].sort_values("proba")[cols].head(10).round(2).to_string(index=False))

In [ ]:
# Where does the model do better or worse? Recall by position among true All-Stars
by_pos = (test[test.all_star == 1].groupby("pos_main")["pred"].agg(["mean", "count"])
          .rename(columns={"mean": "recall", "count": "n_all_stars"}).round(2))
by_pos

## 9. Limitations, Ethics, and Reflection
Discussed in full in the written post. Key points to keep in mind when reading the results:
- **Timing leak-ish issue:** All-Stars are chosen in mid-season, but per-game stats cover the *full* season, including games after selection.
- **Selection is partly a popularity contest** (fan vote), so labels encode popularity, market size, and reputation, not just production.
- **Defense and impact are under-measured** by box-score stats; high-impact defenders and role players are systematically under-predicted.
- **Roster slots are limited:** a very good player can miss out simply because a conference was stacked.
- **Errors matter differently:** a false negative can understate a deserving player's value; this model should *support* discussion, not decide contracts, awards, or playing time.

## 10. Code and Transparency
- **Dataset:** Datta, S. *NBA Stats (1947-present)* [Data set]. Kaggle. Data originally from Basketball-Reference.com (Sports Reference LLC).
- **Libraries:** pandas, NumPy, scikit-learn, matplotlib, seaborn.
- **AI usage disclosure:** fill in per your course policy (tool name, version, and what it was used for).